# Pixel Renderer Benchmark
focus on measuring a benchmark for pixel renderer, not using matplotlib and not opening a display window.

this notebook only measures

```text
320 x 240 pressure grid -> 640 x 480 pixel-by-pixel RGB frame using 2 x 2 scaling
```


In [ ]:
import time

import numpy as np

In [ ]:
WIDTH = 320
HEIGHT = 240
SCALE = 2
PRESSURE_LIMIT = 0.5
WARMUP_FRAMES = 10
MEASURED_FRAMES = 100


In [ ]:
# here i create a fake wave like pressure image for renderer to colour pix by pix
def make_pressure_field(width, height):
    yy, xx = np.mgrid[0:height, 0:width]
    cx = width * 0.32
    cy = height * 0.50
    radius = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2)

    wave = 0.45 * np.sin(radius * 0.18) * np.exp(-radius * 0.018)
    wave += 0.18 * np.sin((xx + yy) * 0.05)
    return wave.astype(np.float32)

In [ ]:
def render_wave_pixel_by_pixel(wave, frame, pressure_limit, scale):
    height, width = wave.shape
    
    # loop through every pressure cell and expand it to a scale x scale output block
    for y in range(height):
        for x in range(width):
            pressure = wave[y, x]
            visibility = int(min(abs(pressure) / pressure_limit * 255.0, 255.0))

            # positive pressure means we have red tint
            if pressure > 0:
                red = 255
                green = 255 - visibility
                blue = 255 - visibility

            # negative pressure means we have blue tint    
            elif pressure < 0:
                red = 255 - visibility
                green = 255 - visibility
                blue = 255

            # else white tint 
            else:
                red = 255
                green = 255
                blue = 255

            out_y = y * scale
            out_x = x * scale
            for dy in range(scale):
                for dx in range(scale):
                    frame[out_y + dy, out_x + dx, 0] = red
                    frame[out_y + dy, out_x + dx, 1] = green
                    frame[out_y + dy, out_x + dx, 2] = blue

    return frame


In [ ]:
def print_results(width, height, scale, warmup_frames, measured_frames, wave, frame, frame_times):
    frame_times = np.asarray(frame_times, dtype=np.float64)
    total_time = float(np.sum(frame_times))
    average_time = total_time / measured_frames
    median_time = float(np.median(frame_times))
    min_time = float(np.min(frame_times))
    max_time = float(np.max(frame_times))
    std_time = float(np.std(frame_times))

    output_width = width * scale
    output_height = height * scale
    pressure_cells_per_frame = width * height
    pixels_per_frame = output_width * output_height
    frames_per_second = measured_frames / total_time
    pixels_per_second = pixels_per_frame * frames_per_second

    print("CPU pixel renderer benchmark:")
    print("workload:")
    print(f"  pressure grid            : {width} x {height}")
    print(f"  output frame             : {output_width} x {output_height} ({scale} x scaling)")
    print(f"  pressure cells/frame     : {pressure_cells_per_frame:,}")
    print(f"  output pixels/frame      : {pixels_per_frame:,}")
    print(f"  measured frames          : {measured_frames:,} (+{warmup_frames:,} warmup)")
    print()
    print("performance:")
    print(f"  total render time        : {total_time:.6f} s")
    print(f"  average frame time       : {average_time * 1e3:.6f} ms")
    print(f"  median frame time        : {median_time * 1e3:.6f} ms")
    print(f"  min / max frame time     : {min_time * 1e3:.6f} / {max_time * 1e3:.6f} ms")
    print(f"  frame time std dev       : {std_time * 1e3:.6f} ms")
    print(f"  render throughput        : {frames_per_second:,.2f} fps")
    print(f"  pixel throughput         : {pixels_per_second / 1e6:,.2f} million pixels/sec")
    print()
    print("sanity check:")
    print(f"  input checksum           : {float(np.sum(wave)):.9f}")
    print(f"  output checksum          : {int(np.sum(frame))}")

    return {
        "average_frame_time_ms": average_time * 1e3,
        "frames_per_second": frames_per_second,
        "million_pixels_per_second": pixels_per_second / 1e6,
        "input_checksum": float(np.sum(wave)),
        "output_checksum": int(np.sum(frame)),
    }


In [ ]:
def run_benchmark(width, height, scale, warmup_frames, measured_frames, pressure_limit):
    if measured_frames <= 0:
        raise ValueError("measured_frames must be greater than 0")
    if scale <= 0:
        raise ValueError("scale must be greater than 0")

    wave = make_pressure_field(width, height)
    frame = np.zeros((height * scale, width * scale, 3), dtype=np.uint8)

    for _ in range(warmup_frames):
        render_wave_pixel_by_pixel(wave, frame, pressure_limit, scale)

    frame_times = []
    for _ in range(measured_frames):
        start = time.perf_counter()
        render_wave_pixel_by_pixel(wave, frame, pressure_limit, scale)
        end = time.perf_counter()
        frame_times.append(end - start)

    return print_results(width, height, scale, warmup_frames, measured_frames, wave, frame, frame_times)


## Benchmark

if you want to see the benchmark, run this cell. this cell will show you a few stats about the performance of pixel by pixel renderer

In [ ]:
results = run_benchmark(
    width=WIDTH,
    height=HEIGHT,
    scale=SCALE,
    warmup_frames=WARMUP_FRAMES,
    measured_frames=MEASURED_FRAMES,
    pressure_limit=PRESSURE_LIMIT,
)
results
